# R3m 历史采样边界核验（CPU only）
原 Notebook 中导入，关闭 GPU，保留现有 HF_TOKEN，Run All 一次。只核验 R3h B-H 的三个 Seed、六个边界；不训练、不评价。通过已有 Store 读取固定历史 Checkpoint，保存小型 RNG 证据。当前 NumPy 版本不会被当作历史版本。

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import uuid

os.environ['CUDA_VISIBLE_DEVICES'] = ''
SOURCE_SHA = 'b31e223c0c1c07650e28357607580ccef974f804'
CODE = Path('/kaggle/working/r3m-sampling-code-' + SOURCE_SHA[:12])
OUTPUT = Path('/kaggle/working/r3m-sampling-evidence-' + uuid.uuid4().hex[:12])
REL = Path('research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m')

def execute(*args):
    return subprocess.run(list(map(str, args)), check=True)

if importlib.util.find_spec('huggingface_hub') is None:
    execute(sys.executable, '-m', 'pip', 'install', '--quiet', 'huggingface_hub>=0.24,<2')
if not os.environ.get('HF_TOKEN', '').strip():
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN').strip()
    except Exception:
        raise RuntimeError('Enable the existing HF_TOKEN secret in the original bingjiangli/minicells Notebook.') from None
if not os.environ.get('HF_TOKEN', '').strip():
    raise RuntimeError('Existing HF_TOKEN is empty')
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS='1', HF_HUB_DISABLE_TELEMETRY='1')
if not (CODE / '.git').exists():
    execute('git', 'init', '--quiet', CODE)
    execute('git', '-C', CODE, 'remote', 'add', 'origin', 'https://github.com/ArcheLabs/mini-cells.git')
    execute('git', '-C', CODE, 'config', 'remote.origin.promisor', 'true')
    execute('git', '-C', CODE, 'config', 'remote.origin.partialclonefilter', 'blob:none')
    execute('git', '-C', CODE, 'sparse-checkout', 'init', '--no-cone')
    execute('git', '-C', CODE, 'sparse-checkout', 'set', '--no-cone',
        '/pyproject.toml', '/src/', '/shared/python/',
        '/research/projects/cellular-language-models/engineering/kaggle-continuation/',
        '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/',
        '/research/projects/cellular-language-models/experiments/006-path-first-clm/',
        '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/',
        '/research/projects/continual-learning/context/04-continual-learning/scripts/research/',
        '/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/')
    execute('git', '-C', CODE, 'fetch', '--depth=1', '--filter=blob:none', 'origin', SOURCE_SHA)
    execute('git', '-C', CODE, 'checkout', '--detach', 'FETCH_HEAD')
actual = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip()
if actual != SOURCE_SHA:
    raise RuntimeError('Sampling audit source mismatch')
print({'source_sha': SOURCE_SHA, 'scope': 'CPU_CHECKPOINT_METADATA_ONLY', 'arm': 'B-H', 'seeds': 3, 'requested_boundaries': 18, 'optimizer_updates': 0}, flush=True)
evidence = OUTPUT / 'CHECKPOINT_BOUNDARY_EVIDENCE.json'
try:
    execute(sys.executable, CODE / REL / 'recover_sampling_boundaries.py', '--output', evidence)
    execute(sys.executable, CODE / REL / 'replay_sampling.py',
            '--windows', CODE / REL / 'registration/artifact-recovery-001/independent-review-001/windows.json',
            '--checkpoint-evidence', evidence, '--output', OUTPUT / 'replay')
finally:
    print('Save Version retains small RNG evidence and replay reports: ' + str(OUTPUT), flush=True)
print('Return CHECKPOINT_BOUNDARY_EVIDENCE.json and replay/SAMPLING_REPLAY_REVIEW.json. Registration remains BLOCKED.')
